In [1]:
import gradio as gr
import fitz
import pandas as pd
import ollama
import json

# --- Logic Functions ---

# Use this CSS string to replicate the mockup's background
custom_css = """
.gradio-container {
    background-color: #f3f4f6 !important;
}
.gr-box, .gr-group {
    background-color: white !important;
    border-radius: 8px !important;
    border: 1px solid #e5e7eb !important;
}
"""

def extract_pdf_text(file_obj):
    if file_obj is None:
        return "Niciun fișier încărcat.", ""

    text_complet = ""
    try:
        doc = fitz.open(file_obj.name)

        for page in doc:
            text_complet += page.get_text()

        doc.close()

        log = f"• PDF procesat cu succes.\n• Pagini detectate: {len(doc)}"
        return text_complet, log

    except Exception as e:
        return f"Eroare la procesarea PDF: {str(e)}", "Eroare sistem."


def analyze_single(jd, cv, model):
    logs = "• Initializing Llama 3.2...\n• Comparing Skill Sets...\n• Generating Reasoning..."
    prompt = f"JD: {jd}\nCV: {cv}\nReturn JSON: {{'score': int, 'reasoning': str}}"
    try:
        response = ollama.generate(model=model, prompt=prompt, format='json', options={'temperature': 0})
        data = json.loads(response['response'])
        return data.get('score', 0), data.get('reasoning', ''), logs
    except Exception as e:
        return 0, f"Error: {e}", logs

def run_batch_logic(job_title, model_name):
    # Simulated data for the table
    data = [
        {"Rank": 1, "Candidate Name": "Alice Brown", "Match Score": 94, "Seniority": "Senior", "Applied Date": "10/26", "Status": "✅"},
        {"Rank": 2, "Candidate Name": "Bob Green", "Match Score": 88, "Seniority": "Mid-Senior", "Applied Date": "10/25", "Status": "✅"},
        {"Rank": 3, "Candidate Name": "Charlie White", "Match Score": 72, "Seniority": "Mid-Level", "Applied Date": "10/27", "Status": "✅"},
        {"Rank": 4, "Candidate Name": "Anen Brown", "Match Score": 79, "Seniority": "Mid-Senior", "Applied Date": "10/26", "Status": "🔶"},
    ]
    df = pd.DataFrame(data).sort_values(by="Match Score", ascending=False)
    df["Rank"] = range(1, len(df) + 1)
    return df, f"• Batch analysis for {job_title} complete.\n• {len(df)} profiles ranked."

# --- UI Layout ---

with gr.Blocks(title="AI MatchPro Suite", theme=gr.Theme.from_hub("hmb/wii"), css=custom_css) as demo:
    gr.Markdown("# **AI MatchPro: Recruitment Intelligence Suite**")

    with gr.Tabs():

        # --- TAB 1: SINGLE APPLICATION ---
        with gr.Tab("Dashboard (Single Analysis)"):
            with gr.Row():
                with gr.Column(scale=1):
                    jd_input = gr.Textbox(label="Job Requirements", lines=8)
                    cv_input = gr.Textbox(label="Candidate CV", lines=8)
                    run_single_btn = gr.Button("Generate Match Score", variant="primary", size="lg")
                    pdf_upload = gr.File(label="Add CV (PDF)", file_types=[".pdf"])
                    add_pdf_btn = gr.Button("Process PDF", variant="secondary")

                with gr.Column(scale=2):
                    score_out = gr.Label(label="MATCH SCORE")
                    reason_out = gr.Textbox(label="Reasoning Analysis", lines=6)
                    log_single = gr.Textbox(label="Tool Log", lines=3, interactive=False)
                    extracted_text_out = gr.Textbox(label="Extracted Text", lines=10)
                    status_log = gr.Textbox(label="Status")
                    model_s = gr.Dropdown(choices=["llama3.2"], value="llama3.2", label="Model")

        # --- TAB 2: BATCH PROCESSING ---
        with gr.Tab("Batch Application Processor"):
            with gr.Row():
                # Left Side: Config & Tests
                with gr.Column(scale=1):
                    batch_job = gr.Dropdown(label="Active Role", choices=["Senior Data Scientist", "React Dev"], value="Senior Data Scientist")
                    batch_jd = gr.Textbox(label="JD Summary", lines=4, interactive=False)

                    with gr.Group():
                        run_batch_btn = gr.Button("Run Batch Analysis", variant="primary")

                # Right Side: Table
                with gr.Column(scale=2):
                    gr.Markdown("### Candidate Pipeline - Ranked by Match Score")
                    batch_table = gr.Dataframe(interactive=False)
                    batch_logs = gr.Textbox(label="System Intelligence Logs", lines=3)
                    model_b = gr.Dropdown(choices=["llama3.2"], value="llama3.2", label="Scoring Model")

    # --- Event Handlers ---
    run_single_btn.click(
        fn=analyze_single,
        inputs=[jd_input, cv_input, model_s],
        outputs=[score_out, reason_out, log_single]
    )

    run_batch_btn.click(
        fn=run_batch_logic,
        inputs=[batch_job, model_b],
        outputs=[batch_table, batch_logs]
    )

    add_pdf_btn.click(
        fn=extract_pdf_text,
        inputs=[pdf_upload],
        outputs=[extracted_text_out, status_log]
    )

if __name__ == "__main__":
    demo.launch(share=True)

/Users/rzv/remote/lab01-rzv1/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(
ERROR:    Exception in ASGI application
Traceback (most recent call last):
  File "/Users/rzv/remote/lab01-rzv1/.venv/lib/python3.9/site-packages/uvicorn/protocols/http/h11_impl.py", line 409, in run_asgi
    result = await app(  # type: ignore[func-returns-value]
  File "/Users/rzv/remote/lab01-rzv1/.venv/lib/python3.9/site-packages/uvicorn/middleware/proxy_headers.py", line 60, in __call__
    return await self.app(scope, receive, send)
  File "/Users/rzv/remote/lab01-rzv1/.venv/lib/python3.9/site-packages/fastapi/applications.py", line 1138, in __call__
    await super().__call__(scope, receive, send)
  File "/Users/rzv/remote/lab01-rzv1/.venv/lib/python3.9/site-packages/starlette/applications.py", line 113, i

Running on local URL:  http://127.0.0.1:7860


ERROR:    Exception in ASGI application
Traceback (most recent call last):
  File "/Users/rzv/remote/lab01-rzv1/.venv/lib/python3.9/site-packages/uvicorn/protocols/http/h11_impl.py", line 409, in run_asgi
    result = await app(  # type: ignore[func-returns-value]
  File "/Users/rzv/remote/lab01-rzv1/.venv/lib/python3.9/site-packages/uvicorn/middleware/proxy_headers.py", line 60, in __call__
    return await self.app(scope, receive, send)
  File "/Users/rzv/remote/lab01-rzv1/.venv/lib/python3.9/site-packages/fastapi/applications.py", line 1138, in __call__
    await super().__call__(scope, receive, send)
  File "/Users/rzv/remote/lab01-rzv1/.venv/lib/python3.9/site-packages/starlette/applications.py", line 113, in __call__
    await self.middleware_stack(scope, receive, send)
  File "/Users/rzv/remote/lab01-rzv1/.venv/lib/python3.9/site-packages/starlette/middleware/errors.py", line 186, in __call__
    raise exc
  File "/Users/rzv/remote/lab01-rzv1/.venv/lib/python3.9/site-packages/st


Could not create share link. Please check your internet connection or our status page: https://status.gradio.app.


2026/05/12 16:26:09 [W] [service.go:132] login to server failed: dial tcp 44.237.78.176:7000: i/o timeout
